### Restart and Run All

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2026
quarter = 2
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:08:12 22:36:26


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [4]:
sql = '''
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC'''
sql = sql % (year,quarter,year,quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2026 AND quarter <= 2) 
OR (year = 2026-1 AND quarter >= 2+1)
ORDER BY year DESC, quarter DESC


In [5]:
dfc = pd.read_sql(sql, conlt)
dfc['Counter'] = 1
dfc_grp = dfc.groupby(['name'], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp['Counter'] == 4]
dfc_grp.shape

(91, 5)

In [6]:
sql = '''
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s-1 AND quarter <= %s - 1) 
OR (year = %s-2 AND quarter >= %s)
ORDER BY year DESC, quarter DESC'''
sql = sql % (year,quarter,year,quarter)
dfp = pd.read_sql(sql, conlt)
dfp['Counter'] = 1
dfp_grp = dfp.groupby(['name'], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp['Counter'] == 4]
dfp_grp.shape

(197, 5)

In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8097,10,"5,497,047",4
1,ACE,8097,10,"809,620",4
2,ADVANC,8097,10,"37,207,829",4
3,AEONTS,8097,10,"3,107,917",4
4,AH,8097,10,"733,651",4


In [8]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","5,497,047","2,639,108",48.01%
1,ACE,2026,Q2,"1,016,788","809,620","207,168",25.59%
2,ADVANC,2026,Q2,"53,532,002","37,207,829","16,324,173",43.87%
3,AH,2026,Q2,"826,918","733,651","93,267",12.71%
4,AIE,2026,Q2,"235,904","221,429","14,475",6.54%


In [9]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM yr_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 90


In [10]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

90

In [11]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO yr_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [13]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","5,497,047","2,639,108",48.01%
1,ACE,2026,Q2,"1,016,788","809,620","207,168",25.59%
2,ADVANC,2026,Q2,"53,532,002","37,207,829","16,324,173",43.87%
3,AH,2026,Q2,"826,918","733,651","93,267",12.71%
5,AIMIRT,2026,Q2,"674,826","990,312","-315,486",-31.86%


In [14]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","5,497,047","2,639,108",48.01%
1,ACE,2026,Q2,"1,016,788","809,620","207,168",25.59%
2,ADVANC,2026,Q2,"53,532,002","37,207,829","16,324,173",43.87%
3,AH,2026,Q2,"826,918","733,651","93,267",12.71%
5,AIMIRT,2026,Q2,"674,826","990,312","-315,486",-31.86%


In [15]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","5,497,047","2,639,108",48.01%
1,ACE,2026,Q2,"1,016,788","809,620","207,168",25.59%
2,ADVANC,2026,Q2,"53,532,002","37,207,829","16,324,173",43.87%
3,AH,2026,Q2,"826,918","733,651","93,267",12.71%
9,ASK,2026,Q2,"667,741","303,499","364,242",120.01%


In [16]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
13,BCP,2026,Q2,"21,707,621","1,862,603","19,845,018",1065.45%
64,SGP,2026,Q2,"4,565,168","808,254","3,756,914",464.82%
61,SCC,2026,Q2,"13,396,405","5,015,628","8,380,777",167.09%
63,SCGP,2026,Q2,"6,029,794","2,874,304","3,155,490",109.78%
17,CENTEL,2026,Q2,"3,443,698","1,745,516","1,698,182",97.29%


In [17]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","5,497,047","2,639,108",48.01%
1,ACE,2026,Q2,"1,016,788","809,620","207,168",25.59%
2,ADVANC,2026,Q2,"53,532,002","37,207,829","16,324,173",43.87%
3,AH,2026,Q2,"826,918","733,651","93,267",12.71%
13,BCP,2026,Q2,"21,707,621","1,862,603","19,845,018",1065.45%


In [18]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","5,497,047","2,639,108",48.01%
1,ACE,2026,Q2,"1,016,788","809,620","207,168",25.59%
2,ADVANC,2026,Q2,"53,532,002","37,207,829","16,324,173",43.87%
3,AH,2026,Q2,"826,918","733,651","93,267",12.71%
13,BCP,2026,Q2,"21,707,621","1,862,603","19,845,018",1065.45%


In [19]:
conlt.commit()
conlt.close()

In [20]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:08:12 22:36:26
